<a href="https://colab.research.google.com/github/bkhn85dn/xac_dinh_va_phan_xu_diem_sai/blob/main/animation_show.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!apt-get install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [5]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML

# 1. Tải dữ liệu
# Đảm bảo bạn đã upload file Final_MultiView_3D.npy vào thư mục gốc của Colab
#data = np.load('Final_MultiView_3D_denoised.npy')
file_name = 'FMVD_denoised_oneeuro.npy'
data = np.load(file_name)
print(f"Dữ liệu đã tải: {data.shape}") # (Frames, Joints, XYZ)

# 2. Định nghĩa các kết nối xương (Symmetry cho 17 joints chuẩn Human3.6M/MotionBERT)
# Các cặp chỉ số (index) nối với nhau để tạo thành khung người
skeleton_edges = [
    (0, 1), (1, 2), (2, 3),      # Chân phải
    (0, 4), (4, 5), (5, 6),      # Chân trái
    (0, 7), (7, 8), (8, 9), (9, 10), # Thân và đầu
    (8, 11), (11, 12), (12, 13), # Tay trái
    (8, 14), (14, 15), (15, 16)  # Tay phải
]

# ==========================================
# 3. Tính toán auto-zoom và giới hạn trục
# ==========================================
# Tính toán min/max của tất cả các khớp trên tất cả các khung hình
mins = np.min(data, axis=(0, 1))
maxs = np.max(data, axis=(0, 1))

# Tìm dải giá trị lớn nhất trên cả 3 trục để giữ cho khung người không bị méo
max_range = np.max(maxs - mins)
center = (maxs + mins) / 2

# Áp dụng auto-zoom: Đặt giới hạn khung vẽ xung quanh dữ liệu, với padding 10%
padding = max_range * 0.1
axes_limits = [center - (max_range / 2) - padding, center + (max_range / 2) + padding]

# 4. Thiết lập khung hình vẽ
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Thiết lập giới hạn trục một cách đồng nhất để giữ tỉ lệ
# Chúng ta hoán đổi: DataX -> PlotX, DataZ -> PlotY, DataY -> PlotZ
ax.set_xlim3d([axes_limits[0][0], axes_limits[1][0]]) # X-width
ax.set_ylim3d([axes_limits[0][2], axes_limits[1][2]]) # Y-depth (hoán đổi với Z data)
ax.set_zlim3d([-axes_limits[1][1], -axes_limits[0][1]]) # Z-height (hoán đổi với Y data)

# Đặt nhãn cho các trục để dễ hình dung
ax.set_xlabel('Chiều rộng (X)')
ax.set_ylabel('Chiều sâu (Z)')
ax.set_zlabel('Chiều cao (Y)')

# Đổi góc nhìn để nhìn thuận mắt hơn
ax.view_init(elev=15., azim=45)

lines = [ax.plot([], [], [], 'o-', lw=3)[0] for _ in skeleton_edges]

def init():
    for line in lines:
        line.set_data([], [])
        line.set_3d_properties([])
    return lines

def update(frame):
    # Lấy tọa độ của khung hình hiện tại
    pose = data[frame]

    # Cập nhật từng đoạn xương
    for i, (j1, j2) in enumerate(skeleton_edges):
        x = [pose[j1, 0], pose[j2, 0]]
        y = [pose[j1, 2], pose[j2, 2]]
        z = [-pose[j1, 1], -pose[j2, 1]]

        # Trong hệ tọa độ Pose Estimation, trục Y thường hướng xuống,
        # ta có thể đảo ngược nếu thấy người bị lộn ngược
        lines[i].set_data(x, y)
        lines[i].set_3d_properties(z)

    ax.set_title(f'Frame: {frame}')
    return lines

# 4. Tạo hiệu ứng chuyển động (Animation)
# Giảm interval nếu muốn chạy nhanh hơn, tăng nếu muốn chậm lại
# Nếu 425 frames quá dài, bạn có thể chọn [::2] để nhảy cách khung hình
ani = FuncAnimation(fig, update, frames=range(0, len(data), 1),
                    init_func=init, blit=True, interval=33)

# Hiển thị trên Colab
plt.close() # Đóng hình tĩnh lại
HTML(ani.to_jshtml())
writer = FFMpegWriter(fps=30, metadata=dict(artist='Me'), bitrate=2000)
print("Đang khởi tạo quá trình xuất video... Vui lòng đợi.")

# Lưu animation thành file mp4
# Chú ý: Quá trình này có thể mất vài phút tùy vào số lượng khung hình (425 frames)
video_output = "output_" + file_name.replace(".npy", ".mp4")
ani.save(video_output, writer=writer)

print(f"Xuất video thành công! File '{video_output}' đã nằm trong thư mục Files của Colab.")

# (Tùy chọn) Tải video về máy tính cá nhân ngay lập tức
from google.colab import files
files.download(video_output)

Dữ liệu đã tải: (425, 17, 3)


Đang khởi tạo quá trình xuất video... Vui lòng đợi.
Xuất video thành công! File 'output_FMVD_denoised_oneeuro.mp4' đã nằm trong thư mục Files của Colab.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>